# 공시정보

# Step1: corp_code 불러오기

In [2]:
import os
from dotenv import load_dotenv

import requests
import zipfile
import io
import xml.etree.ElementTree as ET
import pandas as pd

load_dotenv()
API_KEY = os.getenv("DART_API_KEY")

url = "https://opendart.fss.or.kr/api/corpCode.xml"
params = {
    "crtfc_key": API_KEY
}

# 1) API 호출
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

# 2) ZIP 파일을 메모리에서 바로 열기
with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
    xml_filename = zf.namelist()[0]
    with zf.open(xml_filename) as xml_file:
        xml_bytes = xml_file.read()

# 3) XML 파싱
root = ET.fromstring(xml_bytes)

rows = []
for item in root.findall("list"):
    rows.append({
        "corp_code": item.findtext("corp_code"),
        "corp_name": item.findtext("corp_name"),
        "corp_eng_name": item.findtext("corp_eng_name"),
        "stock_code": item.findtext("stock_code"),
        "modify_date": item.findtext("modify_date"),
    })

# 4) DataFrame으로 변환
df = pd.DataFrame(rows)

# 5) stock_code가 결측치가 아니고, 빈 문자열도 아닌 행만 남기기
df = df[df["stock_code"].notna() & (df["stock_code"].str.strip() != "")]
df

,corp_code,corp_name,corp_eng_name,stock_code,modify_date
1943,00260985,한빛네트,"HanbitNet, Inc.",036720,20170630
1955,00264529,엔플렉스,"Nplex,Inc.",040130,20170630
1956,00358545,동서정보기술,"Dong Seo Information Technology Co., Ltd.",055000,20170630
2694,00231567,애드모바일,,032600,20170630
3764,00359614,리더컴,"LEADER COMM.Co.,Ltd",056140,20170630
...,...,...,...,...,...
117222,00591441,어보브반도체,"ABOV Semiconductor Co.,Ltd.",102120,20251215
117223,00683007,엑스큐어,Xcure Corp.,070300,20251215
117257,01182408,소프트캠프,"SOFTCAMP CO., LTD",258790,20251031
117259,00600013,맵스리얼티,MIRAEASSET MAPS REALTY INVESTMENT COMPANY,094800,20251031


In [2]:
dup_rows = df[df["corp_name"].duplicated(keep=False)].sort_values("corp_name")
dup_rows

,corp_code,corp_name,corp_eng_name,stock_code,modify_date
95188,00181712,SK,SK Inc.,034730,20240328
31420,00144155,SK,"SK Holdings Co., Ltd.",003600,20170630
23487,00104467,국민은행,Kookmin Bank,031150,20170630
98006,00386937,국민은행,KOOKMIN BANK,060000,20250102
38531,00925453,나무기술,"NAMU TECH CO., LTD.",167380,20181211
...,...,...,...,...,...
95442,00159412,한국제지,"HANKUK PAPER MFG CO.,LTD",002300,20240628
106198,00793155,핸디소프트,"HANDYSOFT, Inc.",220180,20250910
4541,00231521,핸디소프트,HandySoft Corp.,032380,20170630
103272,01453670,휴럼,"Hurum Co., Ltd.",353190,20240701


# Step2: 기업개황

In [3]:
url = "https://opendart.fss.or.kr/api/company.json"
params = {
    "crtfc_key": API_KEY,
    "corp_code": "00144155"
}
response = requests.get(url, params=params, timeout=30)
response.json()

{'status': '000',
 'message': '정상',
 'corp_code': '00144155',
 'corp_name': 'SK(주)',
 'corp_name_eng': 'SK Holdings Co., Ltd.',
 'stock_name': 'SK',
 'stock_code': '003600',
 'ceo_nm': '조대식',
 'corp_cls': 'E',
 'jurir_no': '1101110022816',
 'bizr_no': '1168102054',
 'adres': '서울특별시 종로구  종로 26 (서린동)',
 'hm_url': 'www.sk.co.kr',
 'ir_url': '',
 'phn_no': '02-2121-0114',
 'fax_no': '02-2121-1891',
 'induty_code': '64992',
 'est_dt': '19621013',
 'acc_mt': '12'}

# Step3: 공시검색

In [3]:
url = "https://opendart.fss.or.kr/api/list.json"
params = {
    "crtfc_key": API_KEY,
    #"corp_code": "00126380",
    "bgn_de": "20260401",
    "end_de": "20260419",
    "page_no": 1,
    "page_count": 100,
    "last_reprt_at": "Y",
}
response = requests.get(url, params=params, timeout=30)
response.json()

{'status': '000',
 'message': '정상',
 'page_no': 1,
 'page_count': 100,
 'total_count': 40524,
 'total_page': 406,
 'list': [{'corp_code': '00498162',
   'corp_name': '한국주택금융공사',
   'stock_code': '',
   'corp_cls': 'E',
   'report_nm': '채권양도등의등록신청서[한국주택금융공사]',
   'rcept_no': '20260417000388',
   'flr_nm': '한국주택금융공사',
   'rcept_dt': '20260417',
   'rm': ''},
  {'corp_code': '00498162',
   'corp_name': '한국주택금융공사',
   'stock_code': '',
   'corp_cls': 'E',
   'report_nm': '채권양도등의등록신청서[한국주택금융공사]',
   'rcept_no': '20260413001337',
   'flr_nm': '한국주택금융공사',
   'rcept_dt': '20260413',
   'rm': ''},
  {'corp_code': '00148504',
   'corp_name': '한국스탠다드차타드은행',
   'stock_code': '000110',
   'corp_cls': 'E',
   'report_nm': '이중상환청구권부채권등록신청서',
   'rcept_no': '20260415000195',
   'flr_nm': '한국스탠다드차타드은행',
   'rcept_dt': '20260415',
   'rm': ''},
  {'corp_code': '00324548',
   'corp_name': '한국투자신탁운용',
   'stock_code': '',
   'corp_cls': 'E',
   'report_nm': '일괄신고서(집합투자증권-신탁형)(한국투자JP모간미국테크증권자투자신탁(USD)(주식-재

# Step4: 공시서류원본파일

In [ ]:
import requests

url = "https://opendart.fss.or.kr/api/document.xml"
params = {
    "crtfc_key": API_KEY,
    "rcept_no": "20251231800604"
}

res = requests.get(url, params=params, timeout=30)
res.content

# with open("dart_original.zip", "wb") as f:
#     f.write(res.content)

# 정기보고서 주요정보

# Step1: 증자(감자) 현황

In [5]:
import requests

url = "https://opendart.fss.or.kr/api/irdsSttus.json"
params = {
    "crtfc_key": API_KEY,
    "corp_code": "00181712",
    "bsns_year": "2024",
    "reprt_code": "11011"
}

res = requests.get(url, params=params, timeout=30)
res.json()

{'status': '000',
 'message': '정상',
 'list': [{'rcept_no': '20250318001364',
   'corp_cls': 'Y',
   'corp_code': '00181712',
   'corp_name': 'SK',
   'isu_dcrs_de': '-',
   'isu_dcrs_stle': '-',
   'isu_dcrs_stock_knd': '-',
   'isu_dcrs_qy': '-',
   'isu_dcrs_mstvdv_fval_amount': '-',
   'isu_dcrs_mstvdv_amount': '-',
   'stlm_dt': '2024-12-31'}]}

# Step2: 배당에 관한 사항 

In [6]:
import requests

url = "https://opendart.fss.or.kr/api/alotMatter.json"
params = {
    "crtfc_key": API_KEY,
    "corp_code": "00181712",
    "bsns_year": "2023",
    "reprt_code": "11011"
}

res = requests.get(url, params=params, timeout=30)
res.json()

{'status': '000',
 'message': '정상',
 'list': [{'rcept_no': '20240319000757',
   'corp_cls': 'Y',
   'corp_code': '00181712',
   'corp_name': 'SK',
   'se': '주당액면가액(원)',
   'thstrm': '200',
   'frmtrm': '200',
   'lwfr': '200',
   'stlm_dt': '2023-12-31'},
  {'rcept_no': '20240319000757',
   'corp_cls': 'Y',
   'corp_code': '00181712',
   'corp_name': 'SK',
   'se': '(연결)당기순이익(백만원)',
   'thstrm': '-776,798',
   'frmtrm': '1,098,683',
   'lwfr': '1,965,612',
   'stlm_dt': '2023-12-31'},
  {'rcept_no': '20240319000757',
   'corp_cls': 'Y',
   'corp_code': '00181712',
   'corp_name': 'SK',
   'se': '(별도)당기순이익(백만원)',
   'thstrm': '362,974',
   'frmtrm': '544,415',
   'lwfr': '1,499,764',
   'stlm_dt': '2023-12-31'},
  {'rcept_no': '20240319000757',
   'corp_cls': 'Y',
   'corp_code': '00181712',
   'corp_name': 'SK',
   'se': '(연결)주당순이익(원)',
   'thstrm': '-13,941',
   'frmtrm': '19,432',
   'lwfr': '37,010',
   'stlm_dt': '2023-12-31'},
  {'rcept_no': '20240319000757',
   'corp_cls': 'Y',
 

# 주요사항보고서 주요정보

In [7]:
import requests

url = "https://opendart.fss.or.kr/api/mgRs.json"
params = {
    "crtfc_key": API_KEY,
    "corp_code": "00181712",
    "bgn_de": "20150101",
    "end_de": "20231231"
}

res = requests.get(url, params=params, timeout=30)
res.json()

{'status': '000',
 'message': '정상',
 'group': [{'title': '일반사항',
   'list': [{'rcept_no': '20150608000268',
     'corp_cls': 'Y',
     'corp_code': '00181712',
     'corp_name': 'SK',
     'rpt_rcpn': '20150528000671',
     'stn': '흡수합병',
     'bddd': '2015년 04월 20일',
     'ctrd': '2015년 04월 20일',
     'gmtsck_shddstd': '2015년 05월 06일',
     'ap_gmtsck': '2015년 06월 26일',
     'aprskh_pd_bgd': '2015년 06월 26일',
     'aprskh_pd_edd': '2015년 07월 16일',
     'aprskh_prc': '에스케이씨앤씨 주식회사 기명식 보통주식: 230,940원\nSK주식회사 기명식 보통주식: 171,853원\nSK주식회사 기명식 우선주식: 114,536원',
     'mgdt_etc': '2015년 04월 21일\n2015년 06월 11일\n2015년 06월 27일\n2015년 06월 27일∼2015년 07월 27일\n2015년 06월 27일∼2015년 07월 31일\n2015년 08월 01일\n\n2015년 08월 03일\n\n2015년 08월 03일\n2015년 08월 03일\n2015년 08월 14일\n2015년 08월 17일',
     'rt_vl': '에스케이씨앤씨 주식회사 기명식 보통주 : SK주식회사 기명식 보통주 = 1 : 0.7367839\n에스케이씨앤씨 주식회사 기명식 우선주 : SK주식회사 기명식 우선주 =1 : 1.1102438',
     'exevl_int': "(1) 보통주 : 해당사항 없음\n\n(보통주의 경우 합병 당사 회사인 에스케이씨앤씨 주식회사와 SK주식회사 모두 유가증권시장주권상장법인이므로 